# bge-m3 embeddings on Colab

Fallback for when `huggingface.co` cannot be reached from the local machine.
Produces exactly the same files `src/features.py` would have written locally.

**Before running:** upload `data/processed/train.parquet`, `val.parquet` and `test.parquet`
to a Drive folder (default `MyDrive/thai-review-embeddings/processed`).

**After running:** download the six `.npy` files from
`MyDrive/thai-review-embeddings/embeddings` into the local `data/embeddings/`.

Runtime -> Change runtime type -> **T4 GPU** before you start.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/thai-review-embeddings")
PROCESSED_DIR = PROJECT_DIR / "processed"
EMBEDDING_DIR = PROJECT_DIR / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

# Must match src/config.py, otherwise the vectors will not match the report
EMBEDDING_MODEL = "BAAI/bge-m3"
EMBEDDING_NAME = "bge-m3"
EMBEDDING_MAX_LENGTH = 512
SPLITS = ["train", "val", "test"]

# The T4 has 16 GB, so a bigger batch than the local 16 fits. Batch size only
# affects speed: every sequence is padded and masked independently.
BATCH_SIZE = 64

sorted(p.name for p in PROCESSED_DIR.glob("*.parquet"))

In [ ]:
import pandas as pd

splits = {}
for name in SPLITS:
    splits[name] = pd.read_parquet(PROCESSED_DIR / f"{name}.parquet")
    print(f"{name:<6} {len(splits[name]):>6} rows")

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL, model_kwargs={"torch_dtype": "float16"})
model.max_seq_length = EMBEDDING_MAX_LENGTH
model

In [ ]:
import time

sample = splits["train"]["text"].head(200).tolist()
started = time.perf_counter()
model.encode(sample, batch_size=BATCH_SIZE, normalize_embeddings=True)
elapsed = time.perf_counter() - started

total_rows = sum(len(part) for part in splits.values())
print(f"200 reviews in {elapsed:.1f}s -> about {elapsed / 200 * total_rows / 60:.0f} min for all {total_rows:,}")

In [ ]:
import numpy as np

for name in SPLITS:
    target = EMBEDDING_DIR / f"{EMBEDDING_NAME}_{name}.npy"
    if target.exists():
        print(f"{name}: already done, skipping")
        continue

    df = splits[name]
    started = time.perf_counter()
    # encode() sorts by length internally and restores the original order,
    # so short reviews are not padded up to the longest one in the batch
    vectors = model.encode(
        df["text"].tolist(),
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)

    np.save(target, vectors)
    np.save(EMBEDDING_DIR / f"{EMBEDDING_NAME}_{name}_ids.npy", df["id"].to_numpy())
    print(f"{name}: {vectors.shape} in {(time.perf_counter() - started) / 60:.1f} min")

In [ ]:
for name in SPLITS:
    vectors = np.load(EMBEDDING_DIR / f"{EMBEDDING_NAME}_{name}.npy")
    ids = np.load(EMBEDDING_DIR / f"{EMBEDDING_NAME}_{name}_ids.npy")
    norms = np.linalg.norm(vectors, axis=1)
    matches = np.array_equal(ids, splits[name]["id"].to_numpy())
    print(
        f"{name:<6} shape={vectors.shape} nan={int(np.isnan(vectors).sum())} "
        f"norm min={norms.min():.4f} max={norms.max():.4f} ids_match={matches}"
    )

In [ ]:
# A negative review should be closest to something that shares its sentiment,
# otherwise the vectors carry nothing useful for the classifiers
train_vectors = np.load(EMBEDDING_DIR / f"{EMBEDDING_NAME}_train.npy")
val_vectors = np.load(EMBEDDING_DIR / f"{EMBEDDING_NAME}_val.npy")
train, val = splits["train"], splits["val"]

for i in val.index[val["label"] == "negative"][:3]:
    similarities = train_vectors @ val_vectors[i]
    nearest = int(np.argmax(similarities))
    print(f"cos={similarities[nearest]:.3f}  query({val.loc[i, 'label']}): {val.loc[i, 'text'][:70]}")
    print(f"            nearest({train.iloc[nearest]['label']}): {train.iloc[nearest]['text'][:70]}")

## Back on the laptop

Copy the six files into `data/embeddings/`, then carry on with

```
python -m src.train --part embeddings
```

`src/features.py` checks the saved ids against the parquet splits, so a mix-up
between the two machines fails loudly instead of silently shifting the labels.